In [6]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torchvision.models as models
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

ModuleNotFoundError: No module named 'tqdm'

# Prepare Data


In [3]:
train_df = pd.read_csv("/kaggle/input/chexpert/train.csv")

train_df

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223409,CheXpert-v1.0-small/train/patient64537/study2/...,Male,59,Frontal,AP,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,-1.0,0.0,1.0,NaN,NaN,NaN
223410,CheXpert-v1.0-small/train/patient64537/study1/...,Male,59,Frontal,AP,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,0.0,-1.0,NaN,-1.0,NaN,NaN,NaN
223411,CheXpert-v1.0-small/train/patient64538/study1/...,Female,0,Frontal,AP,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223412,CheXpert-v1.0-small/train/patient64539/study1/...,Female,0,Frontal,AP,NaN,NaN,1.0,1.0,NaN,NaN,NaN,-1.0,1.0,0.0,NaN,NaN,NaN,0.0


In [4]:
train_df.describe()

,Age,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
count,223414.000000,22381.0,44839.000000,46203.000000,117778.000000,11944.000000,85956.000000,70622.000000,27608.000000,68443.000000,78934.000000,133211.000000,6492.000000,12194.000000,123217.000000
mean,60.430653,1.0,-0.035795,0.409346,0.848911,0.644508,0.456769,-0.183498,-0.461134,-0.005304,0.206540,0.559706,0.134011,0.688699,0.932680
std,17.820925,0.0,0.718442,0.769323,0.472571,0.691607,0.741785,0.753980,0.828249,0.990244,0.493529,0.648859,0.966183,0.565435,0.283377
min,0.000000,1.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,49.000000,1.0,-1.000000,0.000000,1.000000,1.000000,0.000000,-1.000000,-1.000000,-1.000000,0.000000,0.000000,-1.000000,0.000000,1.000000
50%,62.000000,1.0,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
75%,74.000000,1.0,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000
max,90.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
pneumonia_train_df = train_df[["Path", 'Sex', "Age", "AP/PA","No Finding", "Pneumonia"]].copy()
pneumonia_train_df

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Path,Sex,Age,AP/PA,No Finding,Pneumonia
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,AP,1.0,NaN
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,AP,NaN,NaN
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,AP,NaN,NaN
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,NaN,NaN,NaN
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,AP,NaN,NaN
...,...,...,...,...,...,...
223409,CheXpert-v1.0-small/train/patient64537/study2/...,Male,59,AP,NaN,NaN
223410,CheXpert-v1.0-small/train/patient64537/study1/...,Male,59,AP,NaN,0.0
223411,CheXpert-v1.0-small/train/patient64538/study1/...,Female,0,AP,NaN,NaN
223412,CheXpert-v1.0-small/train/patient64539/study1/...,Female,0,AP,NaN,-1.0


In [6]:
pneumonia_train_df = pneumonia_train_df.loc[((pneumonia_train_df['No Finding'].notna()) | (pneumonia_train_df['Pneumonia'].notna()))].copy()
#pneumonia_train_df = pneumonia_train_df.loc[((pneumonia_train_df['Pneumonia'].notna()))]
pneumonia_train_df

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Path,Sex,Age,AP/PA,No Finding,Pneumonia
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,AP,1.0,NaN
5,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,PA,1.0,NaN
6,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,NaN,1.0,NaN
7,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,PA,1.0,NaN
8,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,NaN,1.0,NaN
...,...,...,...,...,...,...
223395,CheXpert-v1.0-small/train/patient64526/study1/...,Male,55,AP,NaN,-1.0
223402,CheXpert-v1.0-small/train/patient64532/study1/...,Female,52,AP,1.0,NaN
223410,CheXpert-v1.0-small/train/patient64537/study1/...,Male,59,AP,NaN,0.0
223412,CheXpert-v1.0-small/train/patient64539/study1/...,Female,0,AP,NaN,-1.0


In [7]:
counts = pneumonia_train_df['Pneumonia'].value_counts()
print(counts)

Pneumonia
-1.0    18770
 1.0     6039
 0.0     2799
Name: count, dtype: int64


In [8]:
counts = pneumonia_train_df['No Finding'].value_counts()
print(counts)

No Finding
1.0    22381
Name: count, dtype: int64


In [9]:
pneumonia_train_df['Label'] = np.where(
    pneumonia_train_df['No Finding'] == 1, 1,  #healthy patients
    np.where(pneumonia_train_df['Pneumonia'].isin([-1, 1]), 0, np.nan)  # pneumonia patients
)

In [10]:
pneumonia_train_df = pneumonia_train_df[pneumonia_train_df['Label'].notna()].copy()
pneumonia_train_df

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Path,Sex,Age,AP/PA,No Finding,Pneumonia,Label
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,AP,1.0,NaN,1.0
5,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,PA,1.0,NaN,1.0
6,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,NaN,1.0,NaN,1.0
7,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,PA,1.0,NaN,1.0
8,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,NaN,1.0,NaN,1.0
...,...,...,...,...,...,...,...
223391,CheXpert-v1.0-small/train/patient64522/study1/...,Female,21,AP,1.0,NaN,1.0
223395,CheXpert-v1.0-small/train/patient64526/study1/...,Male,55,AP,NaN,-1.0,0.0
223402,CheXpert-v1.0-small/train/patient64532/study1/...,Female,52,AP,1.0,NaN,1.0
223412,CheXpert-v1.0-small/train/patient64539/study1/...,Female,0,AP,NaN,-1.0,0.0


In [11]:
counts = pneumonia_train_df['Label'].value_counts()
print(counts)

Label
0.0    24809
1.0    22381
Name: count, dtype: int64


In [12]:
pneumonia_train_df['Age'] = pneumonia_train_df['Age'].fillna(pneumonia_train_df['Age'].median)
pneumonia_train_df_final = pneumonia_train_df[["Path", 'Sex', "Age", "AP/PA", "Label"]].copy()

In [13]:
pneumonia_train_df_final

,Path,Sex,Age,AP/PA,Label
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,AP,1.0
5,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,PA,1.0
6,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20,NaN,1.0
7,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,PA,1.0
8,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33,NaN,1.0
...,...,...,...,...,...
223391,CheXpert-v1.0-small/train/patient64522/study1/...,Female,21,AP,1.0
223395,CheXpert-v1.0-small/train/patient64526/study1/...,Male,55,AP,0.0
223402,CheXpert-v1.0-small/train/patient64532/study1/...,Female,52,AP,1.0
223412,CheXpert-v1.0-small/train/patient64539/study1/...,Female,0,AP,0.0


In [14]:
pneumonia_train_df_final = pneumonia_train_df_final.dropna()

In [15]:
counts = pneumonia_train_df_final['Label'].value_counts()
print(counts)

Label
0.0    20656
1.0    16974
Name: count, dtype: int64


In [16]:
train_df = pneumonia_train_df_final.copy()

In [17]:
train_df['Sex'] = train_df['Sex'].map({'Male': 0, 'Female': 1})
train_df['AP/PA'] = train_df['AP/PA'].map({'AP': 0, 'PA': 1})

In [18]:
train_df

,Path,Sex,Age,AP/PA,Label
0,CheXpert-v1.0-small/train/patient00001/study1/...,1.0,68,0.0,1.0
5,CheXpert-v1.0-small/train/patient00004/study1/...,1.0,20,1.0,1.0
7,CheXpert-v1.0-small/train/patient00005/study1/...,0.0,33,1.0,1.0
11,CheXpert-v1.0-small/train/patient00006/study1/...,1.0,42,0.0,1.0
18,CheXpert-v1.0-small/train/patient00010/study1/...,1.0,50,1.0,1.0
...,...,...,...,...,...
223391,CheXpert-v1.0-small/train/patient64522/study1/...,1.0,21,0.0,1.0
223395,CheXpert-v1.0-small/train/patient64526/study1/...,0.0,55,0.0,0.0
223402,CheXpert-v1.0-small/train/patient64532/study1/...,1.0,52,0.0,1.0
223412,CheXpert-v1.0-small/train/patient64539/study1/...,1.0,0,0.0,0.0


# Train Test Split

In [19]:
train, test = train_test_split(train_df, test_size=0.2)

In [20]:
train.to_csv('train_clean.csv')
test.to_csv('val_clean.csv')

# Dataloading

In [41]:
class CheXpertDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.copy()
        self.transform = transform

        self.age_mean = self.df['Age'].mean()
        self.age_std = self.df['Age'].std()

        self.df['Age'] = self.df['Age'].fillna(self.age_mean)
        self.df['Sex'] = self.df['Sex'].fillna(0)        
        self.df['AP/PA'] = self.df['AP/PA'].fillna(0)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        age = (row['Age'] - self.age_mean) / (self.age_std + 1e-8)
        
        patient_features = torch.tensor([row['Sex'], age, row['AP/PA']], dtype=torch.float32)
        
        label = torch.tensor(row['Label'], dtype=torch.float32)

        image = Image.open(row['Path'].replace('CheXpert-v1.0-small', r'/kaggle/input/chexpert')).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, patient_features, label


In [42]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet
                         std=[0.229, 0.224, 0.225])
])

In [43]:
train_dataset = CheXpertDataset(train, transform=transform)
val_dataset   = CheXpertDataset(test, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4)


In [44]:
images, tabs, labels = next(iter(train_loader))
print(images.shape)  # [B, 3, 224, 224]
print(tabs.shape)    # [B, 3] -> Sex, Age, AP/PA
print(labels.shape)  # [B]

torch.Size([16, 3, 224, 224])
torch.Size([16, 3])
torch.Size([16])


In [45]:
print(images[:2])

tensor([[[[ 0.9303,  0.8104,  0.5878,  ...,  1.5125,  1.8208,  2.0948],
          [ 0.8618,  0.7762,  0.5878,  ..., -0.3712,  0.1939,  0.9303],
          [ 0.6906,  0.7077,  0.6563,  ..., -1.1418, -0.7822, -0.2342],
          ...,
          [-2.1008, -2.1179, -2.1179,  ..., -1.6898, -1.7925, -1.8610],
          [-2.1008, -2.1179, -2.1179,  ..., -1.5185, -1.7069, -1.8097],
          [-2.1008, -2.1179, -2.1179,  ..., -1.3473, -1.4672, -1.6555]],

         [[ 1.0805,  0.9580,  0.7304,  ...,  1.6758,  1.9909,  2.2710],
          [ 1.0105,  0.9230,  0.7304,  ..., -0.2500,  0.3277,  1.0805],
          [ 0.8354,  0.8529,  0.8004,  ..., -1.0378, -0.6702, -0.1099],
          ...,
          [-2.0182, -2.0357, -2.0357,  ..., -1.5980, -1.7031, -1.7731],
          [-2.0182, -2.0357, -2.0357,  ..., -1.4230, -1.6155, -1.7206],
          [-2.0182, -2.0357, -2.0357,  ..., -1.2479, -1.3704, -1.5630]],

         [[ 1.2980,  1.1759,  0.9494,  ...,  1.8905,  2.2043,  2.4831],
          [ 1.2282,  1.1411,  

In [46]:
print(labels)

tensor([1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.])


# Network

In [88]:
class MultiModalAttentionModel(nn.Module):
    def __init__(self, tab_dim=3, d_model=32, dropout=0.5):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.backbone.fc = nn.Identity()  # [B,512]

        self.img_proj = nn.Sequential(
            nn.Linear(512, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.tab_proj = nn.Sequential(
            nn.Linear(tab_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=2, batch_first=True)
        self.attn_dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, images, tabs):
        img_emb = self.img_proj(self.backbone(images))
        tab_emb = self.tab_proj(tabs.float())

        feats = torch.stack([img_emb, tab_emb], dim=1)
        attn_out, _ = self.attn(feats, feats, feats)
        attn_out = self.attn_dropout(attn_out)
        fused = attn_out.mean(dim=1)

        out = self.classifier(fused)
        return out

# Training

In [89]:
def train_one_epoch(model, loader, optimizer, criterion, device="cuda"):
    model.train()
    total_loss = 0
    all_labels, all_preds = [], []

    for images, tabs, labels in tqdm(loader):
        images, tabs = images.to(device), tabs.to(device)
        labels = labels.to(device=device, dtype=torch.float32).view(-1, 1)


        optimizer.zero_grad()
        outputs = model(images, tabs)          # [B,1]

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).int().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec  = recall_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds)
    auc  = roc_auc_score(all_labels, all_preds)

    return total_loss/len(loader), acc, prec, rec, f1, auc


@torch.no_grad()
def evaluate(model, loader, criterion, device="cuda"):
    model.eval()
    total_loss = 0
    all_labels, all_preds = [], []

    for images, tabs, labels in tqdm(loader):
        images, tabs, labels = images.to(device), tabs.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        outputs = model(images, tabs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).int().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec  = recall_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds)
    auc  = roc_auc_score(all_labels, all_preds)

    return total_loss/len(loader), acc, prec, rec, f1, auc


In [90]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MultiModalAttentionModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

EPOCHS = 5
for epoch in range(EPOCHS):
    print(epoch)
    train_loss, train_acc, train_prec, train_rec, train_f1, train_auc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_prec, val_rec, val_f1, val_auc = evaluate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train: loss={train_loss:.4f}, acc={train_acc:.3f}, prec={train_prec:.3f}, rec={train_rec:.3f}, f1={train_f1:.3f}, auc={train_auc:.3f}")
    print(f"Val:   loss={val_loss:.4f}, acc={val_acc:.3f}, prec={val_prec:.3f}, rec={val_rec:.3f}, f1={val_f1:.3f}, auc={val_auc:.3f}")


0


100%|██████████| 471/471 [00:18<00:00, 26.15it/s]


Epoch 1/5
Train: loss=0.6118, acc=0.630, prec=0.556, rec=0.883, f1=0.682, auc=0.652
Val:   loss=0.4235, acc=0.823, prec=0.780, rec=0.847, f1=0.812, auc=0.825
1


100%|██████████| 471/471 [00:17<00:00, 26.35it/s]


Epoch 2/5
Train: loss=0.4731, acc=0.802, prec=0.760, rec=0.819, f1=0.788, auc=0.803
Val:   loss=0.3921, acc=0.837, prec=0.813, rec=0.829, f1=0.821, auc=0.836
2


...

Epoch 3/5
Train: loss=0.4272, acc=0.833, prec=0.803, rec=0.834, f1=0.819, auc=0.833
Val:   loss=0.3919, acc=0.839, prec=0.827, rec=0.813, f1=0.820, auc=0.837
3


100%|██████████| 471/471 [00:18<00:00, 25.81it/s]


Epoch 4/5
Train: loss=0.3779, acc=0.858, prec=0.838, rec=0.849, f1=0.843, auc=0.857
Val:   loss=0.4041, acc=0.839, prec=0.823, rec=0.819, f1=0.821, auc=0.837
4


100%|██████████| 471/471 [00:17<00:00, 26.31it/s]


Epoch 5/5
Train: loss=0.3237, acc=0.877, prec=0.862, rec=0.867, f1=0.865, auc=0.877
Val:   loss=0.4441, acc=0.839, prec=0.811, rec=0.838, f1=0.824, auc=0.839
